[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/02_preprocessing.ipynb)

# 02 — Verifying the Data Before Optimizing on It

**Question.** Do the simulation artifacts match their contract, and what exactly may the optimizer change?

**Inputs.** The simulation artifacts, read-only. **Outputs.** `data/processed/ue.parquet` and `data/processed/cell.parquet`.
The shell equivalent is `task preprocess`. Nothing here is fitted to the data: no imputation, scaling or filtering.

In [1]:
# Environment: locally, move to the project root; on Colab, clone the repository
# and install what Colab lacks. Extra Hydra overrides come from BAND_TILT_OVERRIDES.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    root = Path("/content/band-tilt")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    missing = [pip for module, pip in COLAB_PACKAGES if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
sys.path.insert(0, str(root))
CONFIG_OVERRIDES = os.environ.get("BAND_TILT_OVERRIDES", "").split()

In [2]:
%load_ext autoreload
%autoreload 2

from functools import partial

import pandas as pd

from src.config import load_config
from src.data import schema
from src.data.build import build_cells, build_ue
from src.data.load import load_artifacts, save
from src.evaluation.export import readable, save_table
from src.utils.plotting import save_fig, setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path("reports/figures/02_preprocessing"))
save_table = partial(
    save_table, in_colab=IN_COLAB, directory=Path("reports/tables/02_preprocessing")
)
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 40)

## 1. Contract Verification

Every check names the source of the bound it enforces.

In [3]:
artifacts = load_artifacts(cfg)
checks = schema.verify(artifacts, cfg)
checks_table = readable(checks)
save_table(checks_table, "verification_checks")
schema.require(checks)  # raises, listing every failed check
checks_table

,Check,source,Holds,violations
0,npz tx_name matches the configured cells,configs/simulation.yaml,True,0
1,npz band_label matches the configured bands,configs/simulation.yaml,True,0
2,npz scenario_id matches the manifest,scenario.json,True,0
3,npz grid matches the manifest grid,scenario.json,True,0
4,npz ue_height_m matches the config,configs/simulation.yaml,True,0
5,ue columns are the declared set,configs/simulation.yaml,True,0
6,npz sinr_db has the shape of rsrp_dbm,radio_map.npz,True,0
7,z equals the configured UE height,configs/simulation.yaml,True,0
8,0 <= tile_row < n_rows,scenario.json,True,0
9,0 <= tile_col < n_cols,scenario.json,True,0


## 2. Decision Variables

One absolute tilt per (cell, band) pair, bounded per band. The optimizer may only move tilts inside these ranges.

In [4]:
cell_table = build_cells(cfg, artifacts)
decision_variables = (
    cell_table.assign(at_upper_bound=cell_table["tilt_baseline_deg"] == cell_table["tilt_max_deg"])
    .groupby("band", observed=True, sort=False)
    .agg(
        cells=("cell", "nunique"),
        baseline=("tilt_baseline_deg", "median"),
        minimum=("tilt_min_deg", "min"),
        maximum=("tilt_max_deg", "max"),
        at_upper_bound=("at_upper_bound", "mean"),
    )
    .reset_index()
    .rename(
        columns={
            "cells": "Cells",
            "baseline": "Current tilt [°]",
            "minimum": "Minimum tilt [°]",
            "maximum": "Maximum tilt [°]",
            "at_upper_bound": "Share of cells at the upper bound",
        }
    )
)
decision_variables = readable(decision_variables)
save_table(decision_variables, "decision_variables")
decision_variables

,Band,Cells,Current tilt [°],Minimum tilt [°],Maximum tilt [°],Share of cells at the upper bound
0,2600 MHz,12,12.0,0.0,15.0,0.0
1,1800 MHz,12,12.0,0.0,15.0,0.0
2,700 MHz,12,12.0,0.0,15.0,0.0


**Observations.** _To be written._

## 3. Processed Tables

The UE table is typed and gets the scenario ID. No row is dropped, including UEs no cell reaches, and no value changes.

In [5]:
ue = build_ue(artifacts)
raw = artifacts.ue

assert len(ue) == len(raw), "preprocessing must not drop rows"

audit = pd.DataFrame(
    [
        ("Rows", len(raw), len(ue)),
        ("Columns", raw.shape[1], ue.shape[1]),
        (
            "Memory [MB]",
            raw.memory_usage(deep=True).sum() / 1e6,
            ue.memory_usage(deep=True).sum() / 1e6,
        ),
    ],
    columns=["Property", "Simulation output (CSV)", "Processed table (Parquet)"],
)
save_table(audit, "preprocessing_audit")

ue_path = save(ue, cfg.data.output.ue_file)
cell_path = save(cell_table, cfg.data.output.cell_file)
pd.testing.assert_frame_equal(pd.read_parquet(ue_path), ue)
pd.testing.assert_frame_equal(pd.read_parquet(cell_path), cell_table)
audit

,Property,Simulation output (CSV),Processed table (Parquet)
0,Rows,10066.0000,10066.0000
1,Columns,8.0000,9.0000
2,Memory [MB],0.6444,0.3124
